# Dataset Analysis

This notebook loads data and prepares image dataset for analysis.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns
import os
import cv2
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Mount Google Drive to access files
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import cv2
import os

image = cv2.imread(r"C:\Users\DELL\Documents\ImageDataset\Normales.jpg")

gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

# threshold to separate beans from white background
_, thresh = cv2.threshold(gray, 200, 255, cv2.THRESH_BINARY_INV)

# remove noise
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(5,5))
thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)

# find contours
contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

os.makedirs("beans", exist_ok=True)

i = 0
for c in contours:

    x,y,w,h = cv2.boundingRect(c)

    if w > 40 and h > 40:  # ignore noise
        crop = image[y:y+h, x:x+w]

        cv2.imwrite(f"beans/bean_{i}.jpg", crop)
        i += 1

print("Total beans detected:", i)

In [ ]:
import cv2
import numpy as np
import os

# Load image
img_path = r"C:\Users\DELL\Documents\ImageDataset\Concha.jpg"
img = cv2.imread(img_path)
if img is None:
    print("Cannot load image - check path")
    exit()

gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
blurred = cv2.GaussianBlur(gray, (7, 7), 0)

# Adaptive threshold (usually works best for beans)
thresh = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                               cv2.THRESH_BINARY_INV, 11, 3)

kernel = np.ones((3,3), np.uint8)
thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=1)

contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

os.makedirs("cropped_beans", exist_ok=True)

count = 0
for cnt in contours:
    area = cv2.contourArea(cnt)
    if area < 300:          # skip very small noise
        continue

    x, y, w, h = cv2.boundingRect(cnt)

    # Bigger padding to avoid cutting beans
    pad = 60                # main fix - increase if still cut

    x1 = max(0, x - pad)
    y1 = max(0, y - pad)
    x2 = min(img.shape[1], x + w + pad)
    y2 = min(img.shape[0], y + h + pad)

    # Skip if crop is too small
    if (x2 - x1) < 60 or (y2 - y1) < 60:
        continue

    crop = img[y1:y2, x1:x2]
    count += 1
    cv2.imwrite(f"cropped_beans/Concha_{count}.jpg", crop)

print(f"Saved {count} beans")

In [ ]:
import cv2
import numpy as np
import os

img_path = r"C:\Users\DELL\Documents\coffeeImageDataset\Normales.jpg"
img = cv2.imread(img_path)
if img is None:
    print("Cannot load image - check path")
    exit()

gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
blurred = cv2.GaussianBlur(gray, (7, 7), 0)

thresh = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                               cv2.THRESH_BINARY_INV, 21, 4)

kernel = np.ones((3,3), np.uint8)
thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=1)

cv2.imwrite(r"C:\Users\DELL\Documents\coffeeImageDataset\debug_thresh9.jpg", thresh)  # ← check this

contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

output_dir = r"C:\Users\DELL\Documents\coffeeImageDataset\cropped"
os.makedirs(output_dir, exist_ok=True)

count = 0
TARGET_SIZE = (224, 224)

for cnt in contours:
    area = cv2.contourArea(cnt)
    if area < 470:          # lowered significantly
        continue

    x, y, w, h = cv2.boundingRect(cnt)

    pad = 65
    x1 = max(0, x - pad)
    y1 = max(0, y - pad)
    x2 = min(img.shape[1], x + w + pad)
    y2 = min(img.shape[0], y + h + pad)

    if (x2 - x1) < 60 or (y2 - y1) < 60:
        continue

    crop = img[y1:y2, x1:x2]
    crop_resized = cv2.resize(crop, TARGET_SIZE, interpolation=cv2.INTER_AREA)

    count += 1
    cv2.imwrite(f"{output_dir}\\Concha_{count}.jpg", crop_resized)

print(f"Saved {count} beans")

In [ ]:
# Visualize sample images from each class
def visualize_dataset(dataset_path, classes, samples_per_class=3):
    fig, axes = plt.subplots(len(classes), samples_per_class,
                             figsize=(15, 5*len(classes)))

    for i, class_name in enumerate(classes):
        class_path = os.path.join(dataset_path, class_name)
        images = os.listdir(class_path)[:samples_per_class]

        for j, img_name in enumerate(images):
            img_path = os.path.join(class_path, img_name)
            img = cv2.imread(img_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            axes[i, j].imshow(img)
            axes[i, j].set_title(f"{class_name}\n{img_name}")
            axes[i, j].axis('off')

    plt.tight_layout()
    plt.show()

# Call the function
visualize_dataset(dataset_path, classes)

In [ ]:
import cv2
import numpy as np
import os

input_folder  = r"C:\Users\DELL\Documents\CBD_Coffee Bean Dataset\PB-II"
output_folder = r"C:\Users\DELL\Documents\CBD Dataset cropped\PB-II"
os.makedirs(output_folder, exist_ok=True)

count_global = 1

for filename in sorted(os.listdir(input_folder)):
    if not filename.lower().endswith(('.jpg', '.jpeg', '.png')):
        continue

    img_path = os.path.join(input_folder, filename)
    img = cv2.imread(img_path)
    if img is None:
        continue

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)

    thresh = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                   cv2.THRESH_BINARY_INV, 11, 2)

    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < 750:  # very low to force detection
            continue

        x, y, w, h = cv2.boundingRect(cnt)
        pad = 50
        x1 = max(0, x - pad)
        y1 = max(0, y - pad)
        x2 = min(img.shape[1], x + w + pad)
        y2 = min(img.shape[0], y + h + pad)

        crop = img[y1:y2, x1:x2]
        if crop.size == 0:
            continue

        crop_resized = cv2.resize(crop, (224, 224))

        out_name = f"A_{count_global}.jpg"
        out_path = os.path.join(output_folder, out_name)
        cv2.imwrite(out_path, crop_resized)

        count_global += 1

print("Finished. Check:", output_folder)

In [ ]:
import instaloader
from itertools import islice

# 1. Initialize Instaloader
L = instaloader.Instaloader(
    download_videos=False,      # We only need images for CNNs
    save_metadata=False,        # Skip extra text files
    post_metadata_txt_pattern='' # Don't save captions
)

# 2. Define your target hashtag
HASHTAG = "greencoffeebeans"

# 3. Download the data
# We use 'islice' to limit the download to the first 100 images
print(f"Starting download for #{HASHTAG}...")
hashtag = instaloader.Hashtag.from_name(L.context, HASHTAG)

for post in islice(hashtag.get_posts(), 100):
    L.download_post(post, target=f"dataset/{HASHTAG}")

print("Download complete!")